# Solutions · 10 · Nonlinear viscoelasticity

Worked solutions to the exercises of [notebook 10](../notebooks/10_nonlinear_viscoelasticity.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engrheo is missing): install it from GitHub
    import engrheo
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engrheo"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engrheo import datasets, constitutive as cm
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

In [2]:
sol = pd.read_csv(datasets.path("polymer_solution_startup.csv"), comment="#")
g, tau = np.array([60.0, 25.0, 8.0]), np.array([0.02, 0.2, 2.0])
gie = cm.modes_from_spectrum(g, tau, "giesekus", alpha=0.25)

## Exercise 1

At low shear rates the first normal-stress coefficient $\Psi_1 = N_1/\dot\gamma^2$ tends to $2\sum g_i\tau_i^2$. Check this with the fitted model and with the data at 0.1 1/s.

In [3]:
psi1_linear = 2 * np.sum(g * tau**2)
st = cm.steady_shear(gie, [0.01, 0.1])
d = sol[sol.shear_rate_1_s == 0.1]
print(f"2 sum g tau^2 = {psi1_linear:.3f} Pa s^2")
print(f"model Psi1 at 0.01 and 0.1 1/s: {st['N1'][0]/0.01**2:.3f}, {st['N1'][1]/0.1**2:.3f} Pa s^2")
print(f"data at 0.1 1/s (end of start-up, t = {d.time_s.iloc[-1]:.0f} s): {d.N1_Pa.iloc[-1]/0.1**2:.3f} Pa s^2")

2 sum g tau^2 = 66.048 Pa s^2
model Psi1 at 0.01 and 0.1 1/s: 66.022, 63.640 Pa s^2
data at 0.1 1/s (end of start-up, t = 32 s): 60.766 Pa s^2


At low rates the first normal-stress coefficient approaches its linear-viscoelastic limit $2\sum g_i\tau_i^2$, which
is dominated by the slowest mode (it weights $\tau^2$). At 0.1 1/s the data are close to that limit; the small
difference is noise plus the fact that the start-up had not quite reached steady state (the slowest mode needs
several times 2 s).

## Exercise 2

At which strain ($\dot\gamma\,t$) does the stress overshoot occur at 10 and 100 1/s? Is it roughly constant?

In [4]:
for rate in (10.0, 100.0):
    d = sol[sol.shear_rate_1_s == rate]
    t_max = d.time_s.iloc[int(np.argmax(d.shear_stress_Pa))]
    print(f"{rate:5.0f} 1/s: stress maximum at t = {t_max:.3g} s, i.e. strain {rate * t_max:.1f}")

   10 1/s: stress maximum at t = 0.451 s, i.e. strain 4.5
  100 1/s: stress maximum at t = 0.0702 s, i.e. strain 7.0


The times of the maxima differ by a factor of about six, but the strains - 4.5 and 7 strain units - are similar:
the overshoot is governed mainly by **strain**, marking the point where the microstructure has been deformed
enough to start yielding. (The data are sampled on a coarse time grid, so the strains are approximate.)

## Exercise 3

**Implement it yourself:** integrate the single-mode UCM start-up equations $\lambda\,d\tau_{xy}/dt = -\tau_{xy} + \eta\dot\gamma$, $\lambda\,d\tau_{xx}/dt = -\tau_{xx} + 2\lambda\dot\gamma\tau_{xy}$ with the explicit Euler method and compare with `cm.startup_shear`.

In [5]:
eta, lam, rate = 100.0, 0.5, 2.0
dt = 1e-3
ts = np.arange(0, 3 + dt / 2, dt)
txy, txx = np.zeros(ts.size), np.zeros(ts.size)
for i in range(ts.size - 1):
    txy[i + 1] = txy[i] + dt / lam * (-txy[i] + eta * rate)
    txx[i + 1] = txx[i] + dt / lam * (-txx[i] + 2 * lam * rate * txy[i])
lib = cm.startup_shear([cm.Mode(eta, lam)], rate, ts)
print(f"max rel. difference: stress {np.max(np.abs(txy[1:]/lib['stress'][1:] - 1)):.1e}, N1 {np.max(np.abs(txx[50:]/lib['N1'][50:] - 1)):.1e}")

max rel. difference: stress 1.0e-03, N1 1.7e-02


Explicit Euler agrees to about the step size divided by the relaxation time - first-order accuracy. The library
uses an adaptive implicit/explicit solver (LSODA) with tight tolerances, which also copes with stiff multi-mode
problems where Euler would need tiny steps.